<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/P0_05_masking_indexing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P0-05: Masking & Indexing

**Difficulty**: 🟢 Easy

**Objective**: Master boolean masks, `masked_fill`, `gather`, and `scatter` — used in attention masks, loss computation, and top-k sampling.

In [1]:
import torch

## Task 1: Boolean Masking

In [ ]:
x = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=torch.float)

# TODO: Create a mask for values > 4
mask = x > 4  # YOUR CODE

# TODO: Zero out values <= 4 (this is basically ReLU with threshold=4)
result = x * (x > 4)  # YOUR CODE — use torch.where or masking

expected = torch.tensor([[0, 0, 0], [0, 5, 6], [7, 8, 9]], dtype=torch.float)
assert torch.equal(result, expected)
print('✅ Task 1 passed!')

✅ Task 1 passed!


## Task 2: Causal Mask (Attention Pattern)

Create the mask that prevents attending to future positions.

In [ ]:
N = 5  # sequence length

# TODO: Create a causal mask — lower triangular matrix of True/False
# Position i can attend to positions 0..i (inclusive)
indices = torch.arange(N)
causal_mask = indices.view(-1, 1) >= indices.view(1, -1) # YOUR CODE — shape (N, N), dtype=torch.bool

# TODO: Apply the mask to random attention scores
scores = torch.randn(N, N)
masked_scores = scores.masked_fill(~causal_mask, -torch.inf)  # Set future positions to -inf using .masked_fill()

print(f'Causal mask:\n{causal_mask.int()}')
print(f'\nMasked scores:\n{masked_scores}')

# Verify: upper triangle (excluding diagonal) should be -inf
upper = masked_scores[~causal_mask]
assert (upper == float('-inf')).all()
print('✅ Task 2 passed!')

Causal mask:
tensor([[1, 0, 0, 0, 0],
        [1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1]], dtype=torch.int32)

Masked scores:
tensor([[ 0.4819,    -inf,    -inf,    -inf,    -inf],
        [-1.4986,  1.0894,    -inf,    -inf,    -inf],
        [-1.1436,  0.2874, -0.2172,    -inf,    -inf],
        [ 0.0045, -0.1391, -0.9456,  0.9021,    -inf],
        [-0.9278,  1.3781, -0.5286, -0.9745,  3.0143]])
✅ Task 2 passed!


## Task 3: Gather (Used in Cross-Entropy Loss)

Given logits `(B, C)` and labels `(B,)`, gather the logit for the correct class.

In [ ]:
B, C = 4, 10  # batch, num_classes
logits = torch.randn(B, C)
labels = torch.tensor([3, 7, 1, 5])  # correct class for each sample

# TODO: Use torch.gather to get the logit for each correct class
# Hint: labels needs shape (B, 1) for gather
correct_logits = torch.gather(logits, 1, labels.view(-1, 1)) # YOUR CODE — shape (B, 1)

# Verify manually
for i in range(B):
    assert correct_logits[i, 0] == logits[i, labels[i]]
print('✅ Task 3 passed!')

✅ Task 3 passed!


## Task 4: Top-K Masking (Sampling Pattern)

In [2]:
logits = torch.randn(1, 20)  # 20 vocab items
k = 5

# TODO: Keep only the top-5 logits, set rest to -inf
# Step 1: Use torch.topk to find top-k values and their indices
# Step 2: Create a mask or use scatter to build filtered logits
top_k_vals, top_k_idx = torch.topk(logits, k, dim=-1)

filtered = torch.full_like(logits, float('-inf'))
# TODO: Use scatter_ to place top-k values back
# YOUR CODE HERE
filtered.scatter_(dim=1, src=top_k_vals, index=top_k_idx)

# Verify: exactly k values should not be -inf
assert (filtered != float('-inf')).sum() == k
print('✅ Task 4 passed!')

✅ Task 4 passed!
